# ROCm + PyTorch (gfx1103) Setup Tutorial

This notebook walks through installing AMD ROCm on Ubuntu 24.04, setting up a Python virtual environment, installing PyTorch with ROCm support, and validating that GPU execution works.

### What you’ll do
1. Install required system packages and development tools.
2. Install ROCm drivers & runtime.
3. Verify the `amdgpu` kernel module is loaded.
4. Create and activate a Python virtual environment.
5. Install ROCm-enabled PyTorch wheels.
6. Run a short GPU verification script.

> ⚠️ **Important:** This notebook runs commands requiring `sudo` access and may require a reboot after ROCm installation. Run it on Ubuntu 24.04 with an AMD GPU compatible with gfx1103.

---

## Prerequisite

Install build tools and development headers.

In [ ]:
# Prerequisite
!sudo apt update
!sudo apt install -y gfortran git ninja-build cmake g++ pkg-config xxd patchelf automake libtool python3-venv python3-dev libegl1-mesa-dev texinfo bison flex

# override arch name
!!export HSA_OVERRIDE_GFX_VERSION=11.0.0
#!echo "export HSA_OVERRIDE_GFX_VERSION=11.0.0" >> ~/.bashrc

## Install RoCm

Install the AMD ROCm stack (this may take a while and require significant disk space).

In [ ]:
# Install RoCm
!wget -q https://repo.radeon.com/amdgpu-install/7.2/ubuntu/noble/amdgpu-install_7.2.70200-1_all.deb
!sudo apt install -y ./amdgpu-install_7.2.70200-1_all.deb
!sudo apt update
!sudo apt install -y python3-setuptools python3-wheel
!sudo usermod -a -G render,video $LOGNAME
!newgrp render video
!sudo apt install -y rocm

## Checking AMDGPU Driver

Check that the required `amdgpu` driver version is installed and loaded.

In [ ]:
# Driver check (skip reinstall if already OK)
REQUIRED_AMDGPU = "1:7.2.70200-2278374.24.04"

import subprocess
import os

try:
    INSTALLED_AMDGPU = subprocess.check_output(
        ["dpkg-query", "-W", "-f=${Version}", "amdgpu"], stderr=subprocess.DEVNULL
    ).decode().strip()
except subprocess.CalledProcessError:
    INSTALLED_AMDGPU = ""

amdgpu_loaded = bool(subprocess.check_output(["lsmod"]).decode().count("amdgpu"))

if INSTALLED_AMDGPU == REQUIRED_AMDGPU and amdgpu_loaded:
    print(f"✔ AMDGPU driver version {INSTALLED_AMDGPU} is installed and module is loaded. Skipping driver reinstall.")
else:
    print("⚠️ AMDGPU driver check failed (version mismatch or module not loaded).\n")
    print(f"Installed version: {INSTALLED_AMDGPU or '<none>'}")
    print(f"Required version:  {REQUIRED_AMDGPU}\n")
    print("To reinstall the driver manually, run the steps below from a text console (Ctrl+Alt+F3):\n")
    print("  sudo apt autoremove -y amdgpu-dkms")
    print("  sudo rm -f /etc/apt/sources.list.d/amdgpu.list")
    print("  sudo rm -rf /var/cache/apt/*")
    print("  sudo apt clean all")
    print("  sudo apt install -y ./amdgpu-install_7.2.70200-1_all.deb")
    print("  sudo apt update")
    print("  sudo apt install -y \"linux-headers-$(uname -r)\" \"linux-modules-extra-$(uname -r)\"")
    print("  sudo apt install -y amdgpu-dkms")
    print("  sudo update-initramfs -u -k all")
    print("  sudo reboot")
    raise SystemExit(1)

## Install `uv` and Activate `uv` Virtual Environment

Install `uv` (a lightweight virtual environment manager) and create/activate a `venv`.

In [ ]:
# Install uv and create/activate venv
!sudo apt update
!sudo apt install -y curl
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv venv venv
print("Run 'source venv/bin/activate' to activate the virtual environment.")

## Install Pre-built Packages

Install PyTorch and related packages from the ROCm nightly wheel index.

In [ ]:
# Install pre-built packages (PyTorch ROCm)
!uv pip install --pre torch torchvision torchaudio \
  --index-url https://rocm.nightlies.amd.com/v2/gfx110X-all/

## Test `RoCm` Availability

Run a quick PyTorch script to validate ROCm availability and compare CPU vs GPU runtime.

In [ ]:
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

import torch
import time

print("ROCm:", torch.version.hip)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Create deterministic tensors
tensor_a_cpu = torch.full((5000, 5000), 2.0, device='cpu')
tensor_b_cpu = torch.full((5000, 5000), 3.0, device='cpu')

# CPU computation
start_time = time.time()
result_cpu = tensor_a_cpu + tensor_b_cpu
cpu_time = time.time() - start_time
print(f"CPU operation took: {cpu_time:.6f} seconds")

# GPU computation
if torch.cuda.is_available():
    tensor_a_gpu = tensor_a_cpu.to('cuda')
    tensor_b_gpu = tensor_b_cpu.to('cuda')

    torch.cuda.synchronize()  # ensure accurate timing
    start_time = time.time()

    result_gpu = tensor_a_gpu + tensor_b_gpu

    torch.cuda.synchronize()  # wait for GPU to finish
    gpu_time = time.time() - start_time
    print(f"GPU operation took: {gpu_time:.6f} seconds")

    # Move GPU result back to CPU for comparison
    result_gpu_cpu = result_gpu.to('cpu')

    # Verify correctness
    if torch.allclose(result_cpu, result_gpu_cpu):
        print("CPU and GPU results match!")
    else:
        print("Results differ!")

ROCm: 7.12.60700
GPU available: False
cpu
CPU operation took: 0.015373 seconds
